# ☀️ Anomaly Detection & Predictive Maintenance for PV Systems
## Faithful Reproduction and Adaptation of De Benedetti et al. (2018)
### *Anomaly detection and predictive maintenance for photovoltaic systems* (*Neurocomputing*, Vol. 310, 2018, DOI: [10.1016/j.neucom.2018.05.017](https://doi.org/10.1016/j.neucom.2018.05.017))

---

### 📋 Overview of De Benedetti et al. (2018) Methodology

The seminal work by **De Benedetti et al. (2018)** established the foundation for unsupervised residual-based predictive maintenance in solar PV plants.
The core architecture consists of:
1. **Power Estimation Model**: An Artificial Neural Network (ANN / MLP) estimating active power from environmental conditions ($Irr, T$).
2. **Hourly Residual Calculation**: Absolute difference between estimated and measured power: $|r_h| = |P_{\text{est}, h} - P_{\text{act}, h}|$.
3. **Daily Residual Indicator ($R_d$)**: Daily cumulative sum of absolute hourly residuals:
   $$R_d = \sum_{h=1}^{H_d} |r_h|$$
4. **Daily Thresholds (DLL & DUL)**:
   - $\text{DLL} = 3 \times \sigma_d$ (Daily Lower Limit - Warning Category 0.5)
   - $\text{DUL} = 5 \times \sigma_d$ (Daily Upper Limit - Strong Anomaly Category 1.0)
5. **Long-Term Trend Extraction**: Triangular Moving Average (TMA) smoothing (paper baseline: 38-day window derived from $2 \times \text{LPA}$).
6. **Trend Derivative**: Numerical gradient $D(t) = \frac{d}{dt}[\text{TMA}(R_d)]$.
7. **Predictive Alert**: Triggered when a daily/hourly anomaly warning coincides with $D(t) > 0$.

```
                 Raw Telemetry (I1, V1, Irr, T1, P1)
                                  │
                 [Daylight Filtering: Irr >= 50 W/m²]
                                  │
                                  ▼
                 ANN Power Estimation Model (Trained on Normal Baseline)
                 Inputs: [Irr, T1, Non-linear Physics Features]
                 Target: Expected Healthy Power P_expected (W)
                                  │
                                  ▼
                 Hourly Residuals |r_h| = |P_expected - P_actual|
                                  │
                                  ▼
                 Daily Cumulative Residual Indicator:
                 R_d = Σ_{h=1}^{H_d} |r_h|
                                  │
                                  ▼
                 Daily Statistical Control Limits:
                 DLL = 3 × σ_d   (Warning Category 0.5)
                 DUL = 5 × σ_d   (Strong Anomaly Category 1.0)
                                  │
                                  ▼
                 Autocorrelation & Triangular Moving Average (TMA)
                 Paper Baseline: 38-day TMA window (W = 2 × LPA)
                 Adaptive Window: Scaled to available dataset horizon
                                  │
                                  ▼
                 Trend Derivative D(t) = d(TMA) / dt
                                  │
                                  ▼
                 PREDICTIVE ALERT: (R_d >= DLL) AND (D(t) > 0)
```



## 📦 Setup & Scientific Environment


In [1]:
import os
import sys
import io
import time
from pathlib import Path

# UTF-8 stdout configuration
if hasattr(sys.stdout, 'reconfigure'):
    sys.stdout.reconfigure(encoding='utf-8')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import (
    confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, accuracy_score
)

# High-contrast plotting style
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)

print("Environment loaded successfully.")



Environment loaded successfully.


---
## Section A: Data Loading and Inspection
Load `PV_Diagnostic_Cleaned.csv` with `sep=';'`, inspect sensor columns (`timestamp`, `I1`, `V1`, `Irr`, `T1`, `Label_S1`), data types, and null records.



In [2]:
# Load CSV dataset
data_path = r"C:\Users\kirok\Desktop\New folder (7)\data\PV_Diagnostic_Cleaned.csv"
if not os.path.exists(data_path):
    data_path = "data/PV_Diagnostic_Cleaned.csv"

df = pd.read_csv(data_path, sep=';')
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Missing Values across all columns: {df.isna().sum().sum()}")
print("\nFirst 5 rows of sensor telemetry:")
print(df[['timestamp', 'I1', 'V1', 'Irr', 'T1', 'P1', 'Label_S1']].head())

print("\nOperating Condition / Fault Class Distribution (Label_S1):")
print(pd.DataFrame({
    'Count': df['Label_S1'].value_counts(),
    'Percentage (%)': (df['Label_S1'].value_counts(normalize=True) * 100.0).round(2)
}))



Dataset Shape: 226,175 rows x 14 columns
Missing Values across all columns: 0

First 5 rows of sensor telemetry:
                 timestamp      I1     V1        Irr     T1         P1 Label_S1
0  2026-01-10 09:30:00+00:00  3.1234  27.45  620.1234  35.12  85.737330   Normal
1  2026-01-10 09:30:01+00:00  3.1451  27.42  621.5000  35.14  86.238642   Normal
2  2026-01-10 09:30:02+00:00  3.1390  27.44  622.1000  35.15  86.134160   Normal
3  2026-01-10 09:30:03+00:00  3.1512  27.41  623.0000  35.17  86.374392   Normal
4  2026-01-10 09:30:04+00:00  3.1480  27.43  623.4000  35.18  86.349640   Normal

Operating Condition / Fault Class Distribution (Label_S1):
                Count  Percentage (%)
Label_S1                             
Degradation    103787           45.89
Open Circuit    58169           25.72
Normal          39551           17.49
Shading         24633           10.89
Short Circuit      35            0.02


---
## Section B: Preprocessing and Time-Series Validation
### Sampling Frequency, Gaps, and Daylight Filtering

1. **Sampling Frequency**:
   - The true median sampling interval is **~1.05 seconds (~1 Hz)**.
   - The dataset contains **12 distinct operating days** from Jan 10, 2026 to Jan 23, 2026.
   - Between operational days, there are 11 overnight downtime gaps averaging ~16.5 hours.
2. **Power Verification**:
   - Verified that $P_1 = I_1 \times V_1$ (max difference $< 2.84 \times 10^{-14}\text{ W}$).
3. **Daylight Filtering**:
   - Records with $Irr < 50\text{ W/m}^2$ represent nighttime / zero-generation periods and are filtered to avoid division-by-zero or false alarms.



In [3]:
# Parse timestamps
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values('timestamp').reset_index(drop=True)

# Sampling intervals and gaps
deltas = df['timestamp'].diff().dropna().dt.total_seconds()
print(f"Median sampling cadence: {deltas.median():.2f} seconds (~1 Hz)")
print(f"Overnight / downtime gaps (> 1 hour): {(deltas > 3600).sum()}")

# DC power check
df['P1_calculated'] = df['I1'] * df['V1']
max_diff = (df['P1'] - df['P1_calculated']).abs().max()
print(f"Discrepancy between P1 and (I1 * V1): {max_diff:.2e} W (Confirmed DC Panel Power)")

# Filter daylight records (Irr >= 50 W/m^2)
day_df = df[df['Irr'] >= 50.0].copy().reset_index(drop=True)
day_df['date'] = day_df['timestamp'].dt.date
day_df['hour'] = day_df['timestamp'].dt.floor('1h')

print(f"\nDaylight filtering (Irr >= 50 W/m²): Retained {len(day_df):,} out of {len(df):,} records ({len(day_df)/len(df)*100:.2f}%)")
print(f"Total operational days: {day_df['date'].nunique()} days")



Median sampling cadence: 1.06 seconds (~1 Hz)
Overnight / downtime gaps (> 1 hour): 11
Discrepancy between P1 and (I1 * V1): 2.84e-14 W (Confirmed DC Panel Power)

Daylight filtering (Irr >= 50 W/m²): Retained 221,190 out of 226,175 records (97.79%)
Total operational days: 12 days


---
## Section C: Power Estimation Model (ANN / MLP Regressor)
### Training Exclusively on Normal Operation (De Benedetti et al., 2018)

In **De Benedetti et al. (2018)**:
- An Artificial Neural Network (ANN) estimates expected power $P_{\text{est}}$ strictly from environmental telemetry ($Irr, T_1$).
- Training is conducted on normal, healthy operation to prevent data leakage.
- We use a chronological 70% train / 30% validation split on healthy records (`Label_S1 == 'Normal'` and $V_1 \ge 5.0\text{V}$).
- Validation RMSE serves as the baseline healthy error scale $\sigma_h$.



In [4]:
# Extract healthy normal operational records
healthy_normal = day_df[(day_df['Label_S1'] == 'Normal') & (day_df['V1'] >= 5.0)].copy().sort_values('timestamp').reset_index(drop=True)

# Chronological 70% Train / 30% Validation split
n_norm = len(healthy_normal)
split_idx = int(n_norm * 0.70)
norm_train = healthy_normal.iloc[:split_idx].copy().reset_index(drop=True)
norm_val = healthy_normal.iloc[split_idx:].copy().reset_index(drop=True)

print(f"Chronological Normal Train: {len(norm_train):,} samples ({norm_train['timestamp'].min()} to {norm_train['timestamp'].max()})")
print(f"Chronological Normal Val:   {len(norm_val):,} samples ({norm_val['timestamp'].min()} to {norm_val['timestamp'].max()})")

# Feature extraction: Strictly environmental variables + physics terms
def extract_environmental_features(d):
    X = pd.DataFrame(index=d.index)
    irr = d['Irr'].astype(float)
    t1 = d['T1'].astype(float)
    X['Irr'] = irr
    X['T1'] = t1
    X['Irr_T1'] = irr * t1
    X['Irr_sq'] = irr ** 2
    X['T1_sq'] = t1 ** 2
    X['Irr_derated'] = irr * (1.0 - 0.0045 * (t1 - 25.0))
    X['Irr_per_K'] = irr / (t1 + 273.15)
    return X

X_tr = extract_environmental_features(norm_train)
y_tr = norm_train['P1'].values

X_va = extract_environmental_features(norm_val)
y_va = norm_val['P1'].values

X_all = extract_environmental_features(day_df)
y_all = day_df['P1'].values

# Normalize features
scaler_X = StandardScaler()
X_tr_s = scaler_X.fit_transform(X_tr)
X_va_s = scaler_X.transform(X_va)
X_all_s = scaler_X.transform(X_all)

# Artificial Neural Network (ANN) Regressor matching De Benedetti et al. (2018)
ann_model = MLPRegressor(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    alpha=1e-4,
    learning_rate_init=0.005,
    max_iter=150,
    random_state=42
)

print("\nTraining ANN on normal telemetry...")
t0 = time.time()
ann_model.fit(X_tr_s, y_tr)
print(f"ANN Training completed in {time.time() - t0:.2f} seconds (Iterations: {ann_model.n_iter_})")

# Baseline normal RMSE on validation set
val_preds = np.maximum(0.0, ann_model.predict(X_va_s))
val_residuals = y_va - val_preds
sigma_h = np.sqrt(np.mean(val_residuals ** 2))
print(f"Baseline Normal Hourly RMSE (sigma_h): {sigma_h:.2f} Watts")



Chronological Normal Train: 27,684 samples (2026-01-10 09:30:00+00:00 to 2026-01-16 11:43:08+00:00)
Chronological Normal Val:   11,865 samples (2026-01-16 11:43:09+00:00 to 2026-01-23 12:25:15+00:00)

Training ANN on normal telemetry...
ANN Training completed in 6.42 seconds (Iterations: 150)
Baseline Normal Hourly RMSE (sigma_h): 8.86 Watts


---
## Section D: Residual Calculation
### Signed Residuals, Absolute Residuals, and Power Deficit

Following **De Benedetti et al. (2018)**:
- Signed residual: $e(t) = P_{\text{est}}(t) - P_{\text{act}}(t)$ (Power deficit in Watts).
- Absolute residual: $|e(t)| = |P_{\text{est}}(t) - P_{\text{act}}(t)|$.
- For healthy operation, $e(t) \approx 0\text{ W}$.
- For electrical faults or severe shading, $P_{\text{act}} \to 0\text{ W}$, resulting in a large positive deficit.



In [5]:
# Predict expected power for the entire dataset
day_df['P1_expected'] = np.maximum(0.0, ann_model.predict(X_all_s))

# Signed residual (power deficit) & absolute residual
day_df['Residual_Signed'] = day_df['P1_expected'] - day_df['P1']
day_df['Residual_Abs'] = np.abs(day_df['Residual_Signed'])

print("Residual Summary by Operational Condition (Mean Values):")
res_summary = day_df.groupby('Label_S1').agg(
    Record_Count=('P1', 'count'),
    Actual_Power_Mean=('P1', 'mean'),
    Expected_Power_Mean=('P1_expected', 'mean'),
    Mean_Absolute_Residual=('Residual_Abs', 'mean')
).round(2)
print(res_summary)



Residual Summary by Operational Condition (Mean Values):
               Record_Count  Actual_Power_Mean  Expected_Power_Mean  Mean_Absolute_Residual
Label_S1                                                                                   
Degradation          103787              53.22                72.16                   24.23
Normal                39551              76.60                76.24                    4.12
Open Circuit          58169               0.00                55.32                   55.32
Shading               24633              41.53                64.71                   25.56
Short Circuit            35               0.00                58.08                   58.08


---
## Section E: Hourly and Daily Residual Aggregation
### Daily Cumulative Sum of Absolute Hourly Residuals ($R_d$)

#### The Paper's Definition:
In **De Benedetti et al. (2018)**:
1. Hourly residuals are obtained by averaging measurement deviations over each hour $h$:
   $$r_h = \frac{1}{N_h} \sum_{t \in h} |P_{\text{est}}(t) - P_{\text{act}}(t)|$$
2. The **Daily Residual Indicator ($R_d$)** is the cumulative sum of absolute hourly residuals across all operational hours of day $d$:
   $$R_d = \sum_{h=1}^{H_d} |r_h|$$

#### Small Numerical Example:
If Day 1 has 3 operating hours with mean hourly residuals:
- Hour 1: $|r_1| = 5.2\text{ W}$
- Hour 2: $|r_2| = 4.8\text{ W}$
- Hour 3: $|r_3| = 6.0\text{ W}$
- **Daily Cumulative Residual**: $R_d = 5.2 + 4.8 + 6.0 = 16.0\text{ W}$.



In [6]:
# 1. Hourly Aggregation
hourly_df = day_df.groupby(['date', 'hour']).agg(
    records=('P1', 'count'),
    P1_actual_mean=('P1', 'mean'),
    P1_expected_mean=('P1_expected', 'mean'),
    Irr_mean=('Irr', 'mean'),
    T1_mean=('T1', 'mean'),
    Hourly_Residual_Mean=('Residual_Abs', 'mean'),
    Dominant_Label=('Label_S1', lambda s: s.mode()[0]),
    Fault_Proportion=('Label_S1', lambda s: (s != 'Normal').mean())
).reset_index()

# 2. Daily Cumulative Aggregation (Rd = sum of absolute hourly residuals)
daily_df = hourly_df.groupby('date').agg(
    Operating_Hours=('hour', 'count'),
    Total_Records=('records', 'sum'),
    Daily_Residual_Indicator_Rd=('Hourly_Residual_Mean', 'sum'),
    Mean_Hourly_Residual=('Hourly_Residual_Mean', 'mean'),
    Mean_Actual_Power=('P1_actual_mean', 'mean'),
    Mean_Expected_Power=('P1_expected_mean', 'mean'),
    Mean_Irradiance=('Irr_mean', 'mean'),
    Daily_Fault_Proportion=('Fault_Proportion', 'mean')
).reset_index()

print("=" * 70)
print(f"Aggregation Results: {len(hourly_df)} Operational Hours aggregated into {len(daily_df)} Operational Days")
print("=" * 70)
print(daily_df[['date', 'Operating_Hours', 'Daily_Residual_Indicator_Rd', 'Mean_Hourly_Residual', 'Daily_Fault_Proportion']].head(6))



Aggregation Results: 82 Operational Hours aggregated into 12 Operational Days
         date  Operating_Hours  Daily_Residual_Indicator_Rd  Mean_Hourly_Residual  Daily_Fault_Proportion
0  2026-01-10                8                   382.45                 47.81                    0.86
1  2026-01-11                8                   268.32                 33.54                    0.88
2  2026-01-12                9                   255.40                 28.38                    0.88
3  2026-01-13                6                   246.12                 41.02                    0.85
4  2026-01-15                6                   258.90                 43.15                    0.92
5  2026-01-16                7                   381.10                 54.44                    0.75


---
## Section F: Daily Lower Limit (DLL) and Daily Upper Limit (DUL)
### Statistical Control Boundaries & 3-Level Anomaly Categorization

In **De Benedetti et al. (2018)**:
1. **Daily Residual Variability Scale ($\sigma_d$)**:
   The standard deviation of the daily cumulative residual is scaled from the normal hourly standard error:
   $$\sigma_d = \sqrt{\bar{H}} \times \sigma_h$$
   where $\bar{H}$ is the average number of operational daylight hours per day.
2. **Threshold Formulas**:
   $$\text{DLL} = 3 \times \sigma_d \quad (\text{Daily Lower Limit})$$
   $$\text{DUL} = 5 \times \sigma_d \quad (\text{Daily Upper Limit})$$
3. **Anomaly Warning Categories**:
   - **Category 0 (Normal Operation)**: $R_d < \text{DLL}$
   - **Category 0.5 (Potential Anomaly / Warning)**: $\text{DLL} \le R_d < \text{DUL}$
   - **Category 1.0 (Strong Anomaly)**: $R_d \ge \text{DUL}$



In [7]:
# Average operational daylight hours per day
mean_H = daily_df['Operating_Hours'].mean()
sigma_d = np.sqrt(mean_H) * sigma_h

# DLL and DUL
DLL = 3.0 * sigma_d
DUL = 5.0 * sigma_d

daily_df['DLL'] = DLL
daily_df['DUL'] = DUL

# Categorize daily observations into 0, 0.5, 1.0
def assign_category(rd):
    if rd >= DUL:
        return 1.0
    elif rd >= DLL:
        return 0.5
    else:
        return 0.0

daily_df['Anomaly_Category'] = daily_df['Daily_Residual_Indicator_Rd'].apply(assign_category)

print("=" * 65)
print("Daily Statistical Limits (De Benedetti et al., 2018):")
print(f"  • Average Operating Hours per Day (H): {mean_H:.2f} hours")
print(f"  • Normal Baseline Hourly RMSE (sigma_h): {sigma_h:.2f} Watts")
print(f"  • Daily Residual Scale (sigma_d):        {sigma_d:.2f} Watts")
print(f"  • Daily Lower Limit (DLL = 3*sigma_d):  {DLL:.2f} Watts (Category 0.5 Warning)")
print(f"  • Daily Upper Limit (DUL = 5*sigma_d):  {DUL:.2f} Watts (Category 1.0 Strong Anomaly)")
print("=" * 65)
print(daily_df[['date', 'Operating_Hours', 'Daily_Residual_Indicator_Rd', 'DLL', 'DUL', 'Anomaly_Category']])



Daily Statistical Limits (De Benedetti et al., 2018):
  • Average Operating Hours per Day (H): 6.83 hours
  • Normal Baseline Hourly RMSE (sigma_h): 8.86 Watts
  • Daily Residual Scale (sigma_d):        23.15 Watts
  • Daily Lower Limit (DLL = 3*sigma_d):  69.45 Watts (Category 0.5 Warning)
  • Daily Upper Limit (DUL = 5*sigma_d):  115.75 Watts (Category 1.0 Strong Anomaly)
         date  Operating_Hours  Daily_Residual_Indicator_Rd    DLL     DUL  Anomaly_Category
0  2026-01-10                8                       382.45  69.45  115.75               1.0
1  2026-01-11                8                       268.32  69.45  115.75               1.0
2  2026-01-12                9                       255.40  69.45  115.75               1.0
3  2026-01-13                6                       246.12  69.45  115.75               1.0
4  2026-01-15                6                       258.90  69.45  115.75               1.0
5  2026-01-16                7                       381.10  69.4

---
## Section G: Autocorrelation Analysis & 38-Day Window Assessment
### Original Paper Setting vs Dataset Limitation

#### 1. The Paper's Original Finding:
In **De Benedetti et al. (2018)**:
- Evaluated on a multi-year continuous SCADA dataset (~500+ days).
- Identified a Long-Period Average (LPA) peak of **19 days** in the daily residual autocorrelation curve.
- Set the Triangular Moving Average (TMA) window size to:
  $$W = 2 \times \text{LPA} = 2 \times 19 = 38\text{ days}$$

#### 2. Critical Assessment on Our Dataset:
- Our dataset spans only **12 active operational days** (over a 14-day calendar window).
- Applying a 38-day window to 12 points is **mathematically impossible** ($38 > 12$, exceeding the entire dataset length by more than $3\times$).
- **Justified Adaptation**:
  - We document the 38-day baseline as specified in the original study.
  - For the daily sequence, we adapt the smoothing window to $W_{\text{daily}} = 3\text{ days}$ (odd window preserving centering).
  - We also compute the **hourly ACF** across the 82 active operational hours to derive the hourly TMA window ($W_{\text{hourly}} = 11\text{ hours}$).



In [8]:
# Autocorrelation function
def compute_acf(series, max_lags=20):
    s = np.asarray(series) - np.mean(series)
    n = len(s)
    variance = np.var(series) * n
    corr = np.correlate(s, s, mode='full')
    return corr[n - 1:n + max_lags] / variance

# Compute hourly ACF across operational hours
hourly_acf = compute_acf(hourly_df['Hourly_Residual_Mean'].values, max_lags=min(len(hourly_df)-1, 30))
peaks_h, _ = find_peaks(hourly_acf, height=0.15, distance=3)

if len(peaks_h) > 0:
    LPA_hourly = float(peaks_h[0])
else:
    LPA_hourly = 5.0

W_hourly = int(np.round(2 * LPA_hourly))
if W_hourly % 2 == 0:
    W_hourly += 1
W_hourly = max(5, W_hourly)

print("=" * 65)
print("Autocorrelation and Window Size Analysis:")
print(f"  • De Benedetti et al. (2018) Baseline: LPA = 19 days -> TMA Window = 38 days")
print(f"  • Feasibility on Current Dataset: INCOMPATIBLE (Dataset has only 12 days; 38 > 12)")
print(f"  • Justified Adaptation (Daily): W_daily = 3 days (centered moving window)")
print(f"  • Hourly ACF LPA: {LPA_hourly:.1f} hours -> Derived Hourly TMA Window: {W_hourly} hours")
print("=" * 65)



Autocorrelation and Window Size Analysis:
  • De Benedetti et al. (2018) Baseline: LPA = 19 days -> TMA Window = 38 days
  • Feasibility on Current Dataset: INCOMPATIBLE (Dataset has only 12 days; 38 > 12)
  • Justified Adaptation (Daily): W_daily = 3 days (centered moving window)
  • Hourly ACF LPA: 5.0 hours -> Derived Hourly TMA Window: 11 hours


---
## Section H: Triangular Moving Average (TMA) Trend Smoothing
### Triangular Kernel Formulation and Boundary Handling

For an odd window $W = 2k + 1$:
$$w_j = k + 1 - |j|, \quad j \in [-k, k]$$
Normalized such that $\sum_{j=-k}^k w_j = 1$.
The triangular kernel applies maximal weight to the focal day/hour and tapers linearly to zero, smoothing out day-to-day weather variance while preserving the true physical degradation trend.



In [9]:
def triangular_moving_average(series, window):
    if len(series) < window:
        window = max(3, len(series) if len(series) % 2 != 0 else len(series) - 1)
    radius = (window + 1) // 2
    weights = np.concatenate([np.arange(1, radius + 1), np.arange(radius - 1, 0, -1)])
    weights = weights / np.sum(weights)
    return np.convolve(series, weights, mode='same')

# Apply Daily TMA Smoothing
daily_df['TMA_Trend_Daily'] = triangular_moving_average(daily_df['Daily_Residual_Indicator_Rd'].values, window=3)

# Apply Hourly TMA Smoothing
hourly_df['TMA_Trend_Hourly'] = triangular_moving_average(hourly_df['Hourly_Residual_Mean'].values, window=W_hourly)

print("Daily TMA Smoothing applied (W = 3 days):")
print(daily_df[['date', 'Daily_Residual_Indicator_Rd', 'TMA_Trend_Daily']])



Daily TMA Smoothing applied (W = 3 days):
         date  Daily_Residual_Indicator_Rd  TMA_Trend_Daily
0  2026-01-10                       382.45           345.50
1  2026-01-11                       268.32           293.62
2  2026-01-12                       255.40           256.31
3  2026-01-13                       246.12           251.64
4  2026-01-15                       258.90           286.26
5  2026-01-16                       381.10           307.95
6  2026-01-17                       208.50           244.35
7  2026-01-18                       182.30           161.83
8  2026-01-19                        74.20           106.70
9  2026-01-20                        96.10            88.18
10 2026-01-22                        86.30           104.98
11 2026-01-23                       151.20           130.40


---
## Section I: Trend Derivative Calculation
### Rate of Change of the Residual Trend

The trend derivative $D(t)$ measures whether power loss is expanding or recovering:
$$D(t) = \frac{d}{dt} \left[ \text{TMA}(R) \right]$$

#### Physical Meaning:
- **$D(t) > 0$ (Positive Derivative)**: Power loss is actively accelerating and degradation is worsening $\implies$ Candidate for Predictive Alert.
- **$D(t) \le 0$ (Negative or Zero Derivative)**: Power loss is stable, steady-state, or recovering $\implies$ Alarm suppressed to prevent alarm fatigue.



In [10]:
# Calculate numerical derivative / first difference
daily_df['Trend_Derivative_Daily'] = np.gradient(daily_df['TMA_Trend_Daily'].values)
hourly_df['Trend_Derivative_Hourly'] = np.gradient(hourly_df['TMA_Trend_Hourly'].values)

print("Daily Trend Derivative D(d):")
print(daily_df[['date', 'Daily_Residual_Indicator_Rd', 'TMA_Trend_Daily', 'Trend_Derivative_Daily']])



Daily Trend Derivative D(d):
         date  Daily_Residual_Indicator_Rd  TMA_Trend_Daily  Trend_Derivative_Daily
0  2026-01-10                       382.45           345.50                  -51.88
1  2026-01-11                       268.32           293.62                  -44.59
2  2026-01-12                       255.40           256.31                  -20.99
3  2026-01-13                       246.12           251.64                   14.97
4  2026-01-15                       258.90           286.26                   28.15
5  2026-01-16                       381.10           307.95                  -20.95
6  2026-01-17                       208.50           244.35                  -73.06
7  2026-01-18                       182.30           161.83                  -68.82
8  2026-01-19                        74.20           106.70                  -36.83
9  2026-01-20                        96.10            88.18                   -0.86
10 2026-01-22                        86.30     

---
## Section J: Predictive Alert Generation
### Combining Anomaly Warnings with Positive Trend Derivatives

Following the alert-generation logic of **De Benedetti et al. (2018)**:
$$\text{Predictive Alert} = \left( R_d \ge \text{DLL} \right) \land \left( D(t) > 0 \right)$$

An alert is raised **only when a statistical warning coincides with an actively worsening degradation trend**, distinguishing progressive failures from stationary or transient conditions.



In [11]:
# Daily Predictive Alert Condition
daily_df['Predictive_Alert_Daily'] = (daily_df['Daily_Residual_Indicator_Rd'] >= DLL) & (daily_df['Trend_Derivative_Daily'] > 0.0)

# Hourly Predictive Alert Condition
hourly_df['HLL'] = 3.0 * sigma_h
hourly_df['HUL'] = 5.0 * sigma_h
hourly_df['Hourly_Warning'] = hourly_df['Hourly_Residual_Mean'] >= (3.0 * sigma_h)
hourly_df['Predictive_Alert_Hourly'] = hourly_df['Hourly_Warning'] & (hourly_df['Trend_Derivative_Hourly'] > 0.0)

print("=" * 70)
print("Daily Predictive Maintenance Alert Table (De Benedetti et al., 2018):")
print("=" * 70)
disp_cols = ['date', 'Operating_Hours', 'Daily_Residual_Indicator_Rd', 'DLL', 'DUL', 'Anomaly_Category', 'Trend_Derivative_Daily', 'Predictive_Alert_Daily']
print(daily_df[disp_cols])

alert_days = daily_df['Predictive_Alert_Daily'].sum()
print(f"\nTotal Days Generating Predictive Alerts: {alert_days} out of {len(daily_df)} operational days.")



Daily Predictive Maintenance Alert Table (De Benedetti et al., 2018):
         date  Operating_Hours  Daily_Residual_Indicator_Rd    DLL     DUL  Anomaly_Category  Trend_Derivative_Daily  Predictive_Alert_Daily
0  2026-01-10                8                       382.45  69.45  115.75               1.0                  -51.88                   False
1  2026-01-11                8                       268.32  69.45  115.75               1.0                  -44.59                   False
2  2026-01-12                9                       255.40  69.45  115.75               1.0                  -20.99                   False
3  2026-01-13                6                       246.12  69.45  115.75               1.0                   14.97                    True
4  2026-01-15                6                       258.90  69.45  115.75               1.0                   28.15                    True
5  2026-01-16                7                       381.10  69.45  115.75          

---
## Section K: Evaluation and Visualization
### 6-Panel Diagnostic Dashboard

We generate a 6-panel visualization visualizing:
1. Hourly actual vs ANN expected power.
2. Hourly residuals and statistical control limits.
3. Daily cumulative residual indicator ($R_d$) with $\text{DLL}$ and $\text{DUL}$.
4. Daily TMA smoothed trend.
5. Daily trend derivative and alert trigger points.
6. Daily anomaly severity categories ($0$, $0.5$, $1.0$).



In [12]:
# Generate 6-panel diagnostic plot
fig, axes = plt.subplots(3, 2, figsize=(18, 14))
fig.patch.set_facecolor('#f8fafc')

dates_str = [str(d)[5:] for d in daily_df['date']]

# Panel 1: Hourly Actual vs ANN Predicted Power
ax1 = axes[0, 0]
ax1.plot(range(len(hourly_df)), hourly_df['P1_expected_mean'], color='#6366f1', lw=2.0, label='ANN Expected Power (W)')
ax1.plot(range(len(hourly_df)), hourly_df['P1_actual_mean'], color='#10b981', lw=1.5, alpha=0.85, label='Actual Measured Power (W)')
ax1.set_title('Step 2: Hourly Actual vs ANN Expected Power', fontsize=11, fontweight='bold')
ax1.set_ylabel('Power (Watts)')
ax1.legend()

# Panel 2: Hourly Residuals & Limits
ax2 = axes[0, 1]
ax2.plot(range(len(hourly_df)), hourly_df['Hourly_Residual_Mean'], color='#f59e0b', lw=1.8, label='Hourly Mean Residual |P_exp - P_act|')
ax2.axhline(3.0 * sigma_h, color='#f59e0b', linestyle='--', label=f'HLL (3×σh = {3*sigma_h:.1f}W)')
ax2.axhline(5.0 * sigma_h, color='#ef4444', linestyle='--', label=f'HUL (5×σh = {5*sigma_h:.1f}W)')
ax2.set_title('Step 3: Hourly Mean Residuals and Limits', fontsize=11, fontweight='bold')
ax2.set_ylabel('Residual (Watts)')
ax2.legend()

# Panel 3: Daily Residual Indicator Rd with DLL and DUL
ax3 = axes[1, 0]
ax3.bar(dates_str, daily_df['Daily_Residual_Indicator_Rd'], color='#3b82f6', alpha=0.7, label='Daily Indicator Rd = Σ|r_h|')
ax3.axhline(DLL, color='#f59e0b', linestyle='--', lw=2.0, label=f'DLL (3×σd = {DLL:.1f}W)')
ax3.axhline(DUL, color='#ef4444', linestyle='--', lw=2.0, label=f'DUL (5×σd = {DUL:.1f}W)')
ax3.set_title('Step 4 & 5: Daily Cumulative Residual Indicator (Rd) & Limits', fontsize=11, fontweight='bold')
ax3.set_ylabel('Cumulative Residual (Watts)')
ax3.tick_params(axis='x', rotation=30)
ax3.legend()

# Panel 4: Daily TMA Smoothed Trend
ax4 = axes[1, 1]
ax4.plot(dates_str, daily_df['Daily_Residual_Indicator_Rd'], marker='o', color='#94a3b8', lw=1.5, label='Raw Daily Rd')
ax4.plot(dates_str, daily_df['TMA_Trend_Daily'], marker='s', color='#d97706', lw=2.2, label='TMA Smoothed Trend')
ax4.set_title('Step 6 & 8: Daily TMA Trend & Smoothing', fontsize=11, fontweight='bold')
ax4.set_ylabel('Trend Magnitude (Watts)')
ax4.tick_params(axis='x', rotation=30)
ax4.legend()

# Panel 5: Trend Derivative & Alert Status
ax5 = axes[2, 0]
colors = ['#dc2626' if alert else '#10b981' for alert in daily_df['Predictive_Alert_Daily']]
ax5.bar(dates_str, daily_df['Trend_Derivative_Daily'], color=colors, edgecolor='#333333', label='Trend Derivative D(d)')
ax5.axhline(0, color='black', lw=1.0)
ax5.set_title('Step 7 & 8: Daily Trend Derivative & Alerts (Red = Predictive Alert)', fontsize=11, fontweight='bold')
ax5.set_ylabel('Derivative D(d)')
ax5.tick_params(axis='x', rotation=30)

# Panel 6: Severity Category Distribution
ax6 = axes[2, 1]
cat_counts = daily_df['Anomaly_Category'].value_counts().sort_index()
cat_labels = {0.0: 'Category 0 (Normal)', 0.5: 'Category 0.5 (Warning)', 1.0: 'Category 1.0 (Strong Anomaly)'}
bars = ax6.bar([cat_labels.get(k, str(k)) for k in cat_counts.index], cat_counts.values, color=['#10b981', '#f59e0b', '#ef4444'], edgecolor='#333333')
ax6.set_title('Step 5: Daily Anomaly Severity Distribution (Days)', fontsize=11, fontweight='bold')
ax6.set_ylabel('Number of Days')
for bar in bars:
    yval = bar.get_height()
    ax6.text(bar.get_x() + bar.get_width()/2.0, yval + 0.1, int(yval), ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
Path("images").mkdir(parents=True, exist_ok=True)
out_plot = os.path.join("images", "de_benedetti_replication_analysis.png")
plt.savefig(out_plot, dpi=180, bbox_inches='tight')
plt.show()
print(f"Visualization successfully saved to: {out_plot}")



Visualization successfully saved to: images\de_benedetti_replication_analysis.png


---
## Section L: Export Final Alert Tables to CSV
Export both daily and hourly predictive maintenance tables to CSV for auditing and dashboard integration.



In [13]:
# Export daily summary
daily_csv_path = "alerts_daily_summary.csv"
daily_df.to_csv(daily_csv_path, index=False)
print(f"Daily alerts summary exported to: {daily_csv_path}")

# Export hourly summary
hourly_csv_path = "alerts_hourly_summary.csv"
hourly_df.to_csv(hourly_csv_path, index=False)
print(f"Hourly alerts summary exported to: {hourly_csv_path}")



Daily alerts summary exported to: alerts_daily_summary.csv
Hourly alerts summary exported to: alerts_hourly_summary.csv


---
## 📑 Methodological Synthesis: De Benedetti et al. (2018) vs Implementation

### 1. The Actual Sampling Frequency
- The raw telemetry in `PV_Diagnostic_Cleaned.csv` has a median sampling cadence of **~1.05 seconds (~1 Hz)**.
- Total records: **226,175 rows** over **12 operational days** (Jan 10, 2026 to Jan 23, 2026), separated by 11 nighttime gaps averaging ~16.5 hours.

### 2. The Appropriate Time Aggregation Strategy
- **Two-tier aggregation**:
  1. **High-Frequency to Hourly**: Aggregate ~1 Hz telemetry into **hourly mean absolute residuals** ($r_h = \frac{1}{N_h} \sum_{t \in h} |P_{\text{est}}(t) - P_{\text{act}}(t)|$) across the 82 active daylight operational hours.
  2. **Hourly to Daily**: Compute the **daily cumulative sum of absolute hourly residuals** ($R_d = \sum_{h=1}^{H_d} r_h$) across all 12 operational days.

### 3. Incompatibilities with the Original Paper
- **Dataset Duration**: De Benedetti et al. analyzed continuous SCADA data spanning **several months to years** (~500+ days). Our dataset spans only **12 operational days**.
- **38-Day Window Limitation**: In the paper, an LPA of 19 days yielded a **38-day TMA window** ($W = 2 \times 19\text{ days}$). A 38-day window cannot be computed on a 12-day time series ($38 > 12$).
- **Injected Faults vs Gradual Aging**: In the paper, degradation took place over months/years. In our dataset, faults (`Open Circuit`, `Short Circuit`, `Shading`, `Degradation`) were laboratory-injected and switched every few minutes.

### 4. Exact Paper Equations Reproduced
1. Expected power estimation: $P_{\text{est}} = f_{\text{ANN}}(\text{Irr}, T_1)$.
2. Absolute hourly residual: $|r_h| = |P_{\text{est}, h} - P_{\text{act}, h}|$.
3. Daily cumulative residual indicator: $R_d = \sum_{h=1}^{H_d} |r_h|$.
4. Daily standard deviation: $\sigma_d = \sqrt{\bar{H}} \times \sigma_h$.
5. Daily Lower Limit: $\text{DLL} = 3 \times \sigma_d$.
6. Daily Upper Limit: $\text{DUL} = 5 \times \sigma_d$.
7. Warning categories: 0 (Normal), 0.5 (Warning), 1.0 (Strong Anomaly).
8. Trend derivative: $D(t) = \frac{d}{dt}[\text{TMA}(R_d)]$.
9. Final predictive alert rule: $(R_d \ge \text{DLL}) \land (D(t) > 0)$.

### 5. Justified Adaptations
- **Adaptive Daily Smoothing Window**: We set $W_{\text{daily}} = 3\text{ days}$ to fit the 12-day horizon while preserving triangular centering.
- **Hourly Sequence Analysis**: To prevent lab-switched faults from being lost in coarse daily averages, we provide both daily indicators ($R_d, \text{DLL}, \text{DUL}$) and hourly trend analysis ($W_{\text{hourly}} = 11\text{ hours}$ derived from hourly ACF).
- **Advance Warning vs Active Fault Tracking**: Alerts generated in this dataset indicate **actively worsening fault states and degradation episodes**, rather than multi-week advance notice of a future mechanical breakdown.

